# Advanced Python `Enum` Aliases — 24 Problems with Worked Solutions

**A complete, executable problem set based on the supplied lesson on aliases.** It expands the lesson from alias identity, lookup, iteration, `__members__`, external status names, and `@unique` into production-quality parsers, compatibility layers, serialization, safeguards, and a tested capstone.

**Use:** Run the notebook top-to-bottom in a Python **3.10+** kernel. Everything uses the Python standard library. Every problem includes a challenge, an explained solution, runnable code, and assertion-based verification. Solutions are deliberately shown after each challenge; cover the solution cells if you want to solve independently.

**Scope:** These examples focus on ordinary Python `Enum` aliases, with separate exercises explaining how `IntEnum` and `Flag` differ. No internet, optional packages, or external data are required.

## Learning goals and key semantics

- **Canonical member:** the first declared name with a given value. Later names with an equal value are aliases pointing to **that same object**.
- `E.NAME` and `E['NAME']` resolve by **member name**; `E(value)` resolves by **member value**. An alias's *name* can differ from its canonical `.name`.
- `list(E)` iterates canonical members only, while `E.__members__` maps **all declared names**, including aliases, to member objects, in declaration order.
- `@unique` rejects aliased names with a `ValueError` at **class creation**, not at compile time.
- **Critical integration distinction:** making `busy = 'running'` supports `Status['busy']`, but **not** `Status('busy')`. For raw external values, use an explicit parser or a carefully designed `_missing_` hook.
- Equal Python values can have surprising consequences (`True == 1`, and `IntEnum` may compare equal to integers or other `IntEnum` classes). Choose member values carefully.

**Terminology:** a declared name is an entry in `__members__`; a unique member is an object returned by iteration. Aliases are additional names, not additional objects.

In [1]:
import enum
import json
from collections import Counter, defaultdict
from dataclasses import dataclass
from itertools import product
from typing import Iterable, Mapping

print('Environment ready: standard library only.')

Environment ready: standard library only.


---

# Part I — Identity, lookup, and canonicalization

Establish the exact semantics before designing an API.

## Problem 01 — Predict identity, equality, and canonical metadata  
**Level:** Intermediate

### Challenge

Create an enum for polygon edge counts with `TRIANGLE = 3`, `RECTANGLE = 4`, `SQUARE = 4`, and `RHOMBUS = 4`. Determine how many *distinct member objects* exist. Check `is`, `==`, `.name`, `.value`, and `id`. Explain why a lookup through `SQUARE` reports a canonical name.

### Solution 01: reasoning

`RECTANGLE` is the first name assigned `4`, so the other two names resolve to its object. Identity (`is`) demonstrates that aliases are not extra members. The member stores its first/canonical name, which is why `SQUARE.name` becomes `'RECTANGLE'`.

In [2]:
class PolygonSides(enum.Enum):
    TRIANGLE = 3
    RECTANGLE = 4
    SQUARE = 4
    RHOMBUS = 4

print('Declared names:', tuple(PolygonSides.__members__))
print('Unique members:', list(PolygonSides))
print('Alias name resolves to:', PolygonSides.SQUARE.name)
print('Object IDs for the value 4:', [id(PolygonSides[name]) for name in ('RECTANGLE', 'SQUARE', 'RHOMBUS')])

Declared names: ('TRIANGLE', 'RECTANGLE', 'SQUARE', 'RHOMBUS')
Unique members: [<PolygonSides.TRIANGLE: 3>, <PolygonSides.RECTANGLE: 4>]
Alias name resolves to: RECTANGLE
Object IDs for the value 4: [2329873481104, 2329873481104, 2329873481104]


### Verification 01

Execute the next cell. It uses assertions so a silent pass means the requirements hold.

In [3]:
assert PolygonSides.RECTANGLE is PolygonSides.SQUARE is PolygonSides.RHOMBUS
assert PolygonSides.RECTANGLE == PolygonSides.SQUARE
assert PolygonSides.SQUARE.name == 'RECTANGLE'
assert PolygonSides.SQUARE.value == 4
assert len(PolygonSides.__members__) == 4
assert len(list(PolygonSides)) == 2
print('PASS 01 — identity and canonical metadata')

PASS 01 — identity and canonical metadata


**Engineering note:** Never use enum aliases when the different names must carry independent metadata or identity; model separate objects in that case.

## Problem 02 — Name lookup versus value lookup  
**Level:** Intermediate

### Challenge

Show results for `PolygonSides['SQUARE']`, `PolygonSides(4)`, and `PolygonSides['TRIANGLE']`. Capture the exceptions raised by a nonexistent name and nonexistent value. Build a small safe lookup function that makes the choice of lookup mechanism explicit.

### Solution 02: reasoning

Square brackets invoke **name** lookup and raise `KeyError` for an unknown name. Calling an enum invokes **value** lookup and raises `ValueError` for an unknown value. Both valid routes return the same canonical object for the four-sided polygon.

In [4]:
def polygon_from_name(name: str) -> PolygonSides:
    """Resolve a declared symbolic name, including aliases."""
    return PolygonSides[name]


def polygon_from_value(value: int) -> PolygonSides:
    """Resolve a stored edge count."""
    return PolygonSides(value)

for label, operation in [('unknown name', lambda: PolygonSides['HEXAGON']),
                         ('unknown value', lambda: PolygonSides(6))]:
    try:
        operation()
    except (KeyError, ValueError) as exc:
        print(f'{label}: {type(exc).__name__}')

unknown name: KeyError
unknown value: ValueError


### Verification 02

Execute the next cell. It uses assertions so a silent pass means the requirements hold.

In [5]:
assert polygon_from_name('SQUARE') is PolygonSides.RECTANGLE
assert polygon_from_value(4) is PolygonSides.RECTANGLE
assert polygon_from_name('TRIANGLE') is PolygonSides.TRIANGLE
for callback, error in [(lambda: polygon_from_name('HEXAGON'), KeyError),
                        (lambda: polygon_from_value(6), ValueError)]:
    try:
        callback()
    except error:
        pass
    else:
        raise AssertionError(f'Expected {error.__name__}')
print('PASS 02 — lookup modes and exception contracts')

PASS 02 — lookup modes and exception contracts


## Problem 03 — Iteration, size, and full declared-name inspection  
**Level:** Intermediate

### Challenge

Produce (a) unique members, (b) every declared name, (c) a name-to-canonical-name mapping, and (d) the alias-only names. Verify that iteration omits aliases without losing their availability via `__members__`.

### Solution 03: reasoning

Iteration yields each unique member once. `__members__` is a read-only mapping of **all names** to their member objects; an alias can be identified when the mapping key is not equal to the object's canonical `.name`.

In [6]:
unique_polygons = tuple(PolygonSides)
declared_polygon_names = tuple(PolygonSides.__members__)
canonical_by_name = {name: member.name for name, member in PolygonSides.__members__.items()}
polygon_aliases = {name: member.name for name, member in PolygonSides.__members__.items()
                   if name != member.name}
print('Unique:', unique_polygons)
print('Canonical by declared name:', canonical_by_name)
print('Aliases only:', polygon_aliases)

Unique: (<PolygonSides.TRIANGLE: 3>, <PolygonSides.RECTANGLE: 4>)
Canonical by declared name: {'TRIANGLE': 'TRIANGLE', 'RECTANGLE': 'RECTANGLE', 'SQUARE': 'RECTANGLE', 'RHOMBUS': 'RECTANGLE'}
Aliases only: {'SQUARE': 'RECTANGLE', 'RHOMBUS': 'RECTANGLE'}


### Verification 03

Execute the next cell. It uses assertions so a silent pass means the requirements hold.

In [7]:
assert tuple(m.name for m in unique_polygons) == ('TRIANGLE', 'RECTANGLE')
assert declared_polygon_names == ('TRIANGLE', 'RECTANGLE', 'SQUARE', 'RHOMBUS')
assert canonical_by_name['SQUARE'] == 'RECTANGLE'
assert polygon_aliases == {'SQUARE': 'RECTANGLE', 'RHOMBUS': 'RECTANGLE'}
assert len(PolygonSides) == 2
print('PASS 03 — iteration and complete name mapping')

PASS 03 — iteration and complete name mapping


## Problem 04 — Prove `__members__` is read-only  
**Level:** Intermediate

### Challenge

Attempt to add `HEXAGON` through `PolygonSides.__members__` without letting the expected error interrupt the notebook. Verify that no member was created. Explain why a read-only view matters to consumers.

### Solution 04: reasoning

`__members__` is a `mappingproxy`, not a mutable dictionary. Mutating it raises `TypeError`. Consumers may inspect the map without casually changing the enum's declared names at runtime.

In [8]:
try:
    PolygonSides.__members__['HEXAGON'] = PolygonSides.TRIANGLE
except TypeError as exc:
    print('Mutation blocked:', type(exc).__name__)
else:
    raise AssertionError('The member mapping should be read-only')

Mutation blocked: TypeError


### Verification 04

Execute the next cell. It uses assertions so a silent pass means the requirements hold.

In [9]:
assert 'HEXAGON' not in PolygonSides.__members__
assert 'SQUARE' in PolygonSides.__members__
assert isinstance(PolygonSides.__members__, type(type.__dict__)) or hasattr(PolygonSides.__members__, 'keys')
print('PASS 04 — read-only member mapping')

PASS 04 — read-only member mapping


---

# Part II — Structural alias analysis

Build reusable introspection and robust checks instead of relying on manual inspection.

## Problem 05 — Build an alias inventory with declaration order  
**Level:** Advanced

### Challenge

Write `alias_groups(enum_type)` returning a dictionary from canonical name to an ordered tuple of all declared names for that member, including the canonical name itself. Do not assume numeric values or sort the names.

### Solution 05: reasoning

Group by member object rather than value. Each alias maps to the canonical object and `.name` is a stable, human-readable grouping key. Iterating `__members__` preserves declaration order.

In [10]:
def alias_groups(enum_type: type[enum.Enum]) -> dict[str, tuple[str, ...]]:
    groups: dict[str, list[str]] = {}
    for declared_name, member in enum_type.__members__.items():
        groups.setdefault(member.name, []).append(declared_name)
    return {canonical: tuple(names) for canonical, names in groups.items()}

polygon_groups = alias_groups(PolygonSides)
print(polygon_groups)

{'TRIANGLE': ('TRIANGLE',), 'RECTANGLE': ('RECTANGLE', 'SQUARE', 'RHOMBUS')}


### Verification 05

Execute the next cell. It uses assertions so a silent pass means the requirements hold.

In [11]:
assert polygon_groups == {
    'TRIANGLE': ('TRIANGLE',),
    'RECTANGLE': ('RECTANGLE', 'SQUARE', 'RHOMBUS'),
}
assert all(names[0] == canonical for canonical, names in polygon_groups.items())
print('PASS 05 — alias inventory')

PASS 05 — alias inventory


## Problem 06 — Produce a lossless declared-name audit report  
**Level:** Advanced

### Challenge

Create a list of records with fields `declared`, `canonical`, `value`, `is_alias`. Count the aliases separately from the unique members. Explain why a list of `list(Enum)` alone cannot serve as a complete audit record.

### Solution 06: reasoning

A simple enum iteration discards alias *names*. An audit report must traverse `__members__.items()` to preserve the user-visible spellings and their canonical relationships.

In [12]:
def enum_audit(enum_type: type[enum.Enum]) -> list[dict[str, object]]:
    return [
        {'declared': name, 'canonical': member.name,
         'value': member.value, 'is_alias': name != member.name}
        for name, member in enum_type.__members__.items()
    ]

polygon_audit = enum_audit(PolygonSides)
print(json.dumps(polygon_audit, indent=2))

[
  {
    "declared": "TRIANGLE",
    "canonical": "TRIANGLE",
    "value": 3,
    "is_alias": false
  },
  {
    "declared": "RECTANGLE",
    "canonical": "RECTANGLE",
    "value": 4,
    "is_alias": false
  },
  {
    "declared": "SQUARE",
    "canonical": "RECTANGLE",
    "value": 4,
    "is_alias": true
  },
  {
    "declared": "RHOMBUS",
    "canonical": "RECTANGLE",
    "value": 4,
    "is_alias": true
  }
]


### Verification 06

Execute the next cell. It uses assertions so a silent pass means the requirements hold.

In [13]:
assert len(polygon_audit) == 4
assert sum(row['is_alias'] for row in polygon_audit) == 2
assert polygon_audit[2] == {'declared': 'SQUARE', 'canonical': 'RECTANGLE',
                            'value': 4, 'is_alias': True}
assert len(list(PolygonSides)) == 2
print('PASS 06 — lossless audit report')

PASS 06 — lossless audit report


## Problem 07 — Generic alias detector and uniqueness invariant  
**Level:** Advanced

### Challenge

Write `has_aliases()` and `assert_no_aliases()` without relying on private enum attributes. The latter should raise a clear `ValueError` listing unexpected aliases. Test both a duplicate-valued and a genuinely unique enum.

### Solution 07: reasoning

One public invariant is `len(__members__) == len(enum_type)`. A more useful error enumerates the names whose dictionary keys differ from `.name`. This detects aliases even when values are strings or tuples.

In [14]:
def has_aliases(enum_type: type[enum.Enum]) -> bool:
    return len(enum_type.__members__) != len(enum_type)


def assert_no_aliases(enum_type: type[enum.Enum]) -> None:
    offenders = [name for name, member in enum_type.__members__.items()
                 if name != member.name]
    if offenders:
        raise ValueError(f'{enum_type.__name__} aliases are not allowed: {offenders}')

class UniqueDirection(enum.Enum):
    NORTH = 'N'
    SOUTH = 'S'
    EAST = 'E'
    WEST = 'W'

assert_no_aliases(UniqueDirection)

### Verification 07

Execute the next cell. It uses assertions so a silent pass means the requirements hold.

In [15]:
assert has_aliases(PolygonSides) is True
assert has_aliases(UniqueDirection) is False
try:
    assert_no_aliases(PolygonSides)
except ValueError as exc:
    assert 'SQUARE' in str(exc) and 'RHOMBUS' in str(exc)
else:
    raise AssertionError('Aliased enum was not rejected')
print('PASS 07 — generic alias checks')

PASS 07 — generic alias checks


## Problem 08 — Design an explicit preferred-name display policy  
**Level:** Advanced

### Challenge

A legacy API accepts `SQUARE` but UI text should say `four-sided`. Implement `preferred_label()` using a mapping keyed by canonical member, with a fallback. Demonstrate that alias lookup does not retain which name the caller originally supplied.

### Solution 08: reasoning

All aliases have the same identity, so the member alone cannot reveal whether the input spelling was `RECTANGLE` or `SQUARE`. Use a separate label map for UI display; retain the raw input separately if provenance matters.

In [16]:
POLYGON_LABELS = {
    PolygonSides.TRIANGLE: 'three-sided',
    PolygonSides.RECTANGLE: 'four-sided',
}

def preferred_label(member: PolygonSides) -> str:
    return POLYGON_LABELS.get(member, member.name.replace('_', ' ').lower())

requested_spelling = 'SQUARE'
parsed_polygon = PolygonSides[requested_spelling]
print({'raw': requested_spelling, 'canonical': parsed_polygon.name,
       'display': preferred_label(parsed_polygon)})

{'raw': 'SQUARE', 'canonical': 'RECTANGLE', 'display': 'four-sided'}


### Verification 08

Execute the next cell. It uses assertions so a silent pass means the requirements hold.

In [17]:
assert preferred_label(PolygonSides.SQUARE) == 'four-sided'
assert preferred_label(PolygonSides.RHOMBUS) == 'four-sided'
assert PolygonSides['SQUARE'].name == 'RECTANGLE'
assert requested_spelling != parsed_polygon.name
print('PASS 08 — explicit presentation mapping')

PASS 08 — explicit presentation mapping


---

# Part III — Integrating two external status systems

The supplied source combines `ready / busy / finished_no_error / finished_with_errors` and `ready / processing / ran_ok / errored`. Make that approach safe at an actual system boundary.

## Problem 09 — Implement the source-style status aliases correctly  
**Level:** Intermediate

### Challenge

Define the four canonical statuses `ready`, `running`, `ok`, and `errors` as the first names. Assign names from the two source systems as aliases, then verify the names and number of distinct states. Check the meaning of `ExternalStatus['busy']` versus `ExternalStatus('busy')`.

### Solution 09: reasoning

Aliases are created by **equal assigned values**. Here `busy` stores `'running'`, not `'busy'`. Therefore lookup by the declared name works, but lookup by raw string value `'busy'` raises `ValueError`. The two operations must not be conflated.

In [18]:
class ExternalStatus(enum.Enum):
    ready = 'ready'
    running = 'running'
    busy = 'running'
    processing = 'running'
    ok = 'ok'
    finished_no_error = 'ok'
    ran_ok = 'ok'
    errors = 'errors'
    finished_with_errors = 'errors'
    errored = 'errors'

print('Canonical states:', [member.name for member in ExternalStatus])
print('Legacy "busy" by name:', ExternalStatus['busy'])
try:
    ExternalStatus('busy')
except ValueError:
    print('Raw value "busy" is NOT a stored value; name lookup is needed.')

Canonical states: ['ready', 'running', 'ok', 'errors']
Legacy "busy" by name: ExternalStatus.running
Raw value "busy" is NOT a stored value; name lookup is needed.


### Verification 09

Execute the next cell. It uses assertions so a silent pass means the requirements hold.

In [19]:
assert [m.name for m in ExternalStatus] == ['ready', 'running', 'ok', 'errors']
assert ExternalStatus['busy'] is ExternalStatus.running
assert ExternalStatus['processing'] is ExternalStatus.running
assert ExternalStatus['ran_ok'] is ExternalStatus.ok
assert ExternalStatus['errored'] is ExternalStatus.errors
assert len(ExternalStatus.__members__) == 10
try:
    ExternalStatus('busy')
except ValueError:
    pass
else:
    raise AssertionError('Expected value lookup to reject "busy"')
print('PASS 09 — source-style aliases and the critical lookup distinction')

PASS 09 — source-style aliases and the critical lookup distinction


## Problem 10 — Write a strict parser for upstream status strings  
**Level:** Advanced

### Challenge

Create `parse_status_name(raw)` that accepts declared upstream names and returns an `ExternalStatus`. Reject nonstrings, misspellings, and unrecognized input with a consistent `ValueError` mentioning the input. Do **not** change capitalization silently.

### Solution 10: reasoning

A boundary parser can validate type and call name lookup. Catch `KeyError` narrowly and raise `ValueError` with an application-specific message. This avoids accepting ambiguous or unwanted spellings by accident.

In [20]:
def parse_status_name(raw: object) -> ExternalStatus:
    if not isinstance(raw, str):
        raise ValueError(f'Expected a status string; got {raw!r}')
    try:
        return ExternalStatus[raw]
    except KeyError as exc:
        raise ValueError(f'Unrecognized status: {raw!r}') from exc

for raw in ('ready', 'busy', 'processing', 'ran_ok', 'errored'):
    print(raw, '->', parse_status_name(raw).name)

ready -> ready
busy -> running
processing -> running
ran_ok -> ok
errored -> errors


### Verification 10

Execute the next cell. It uses assertions so a silent pass means the requirements hold.

In [21]:
assert parse_status_name('busy') is ExternalStatus.running
assert parse_status_name('finished_with_errors') is ExternalStatus.errors
for bad in (None, 123, ' Busy', 'BUSY', 'unexpected'):
    try:
        parse_status_name(bad)
    except ValueError as exc:
        assert repr(bad) in str(exc)
    else:
        raise AssertionError(f'Accepted invalid input: {bad!r}')
print('PASS 10 — strict boundary parser')

PASS 10 — strict boundary parser


## Problem 11 — Batch-normalize records while preserving provenance  
**Level:** Advanced

### Challenge

Input records have `system`, `job_id`, and `status` strings. Produce new records containing `canonical_status`, `raw_status`, and an alias flag without mutating input. On unknown input, include the job identifier in a raised error. Retaining source spelling is mandatory for debugging.

### Solution 11: reasoning

Canonicalize with the strict parser while separately keeping the original string. `member.name` is canonical, whereas `raw_status` records which legacy spelling occurred. Copy each dictionary to avoid silently changing caller-owned data.

In [22]:
def normalize_records(records: Iterable[Mapping[str, object]]) -> list[dict[str, object]]:
    result = []
    for record in records:
        try:
            member = parse_status_name(record['status'])
        except (KeyError, ValueError) as exc:
            raise ValueError(f"Invalid status for job {record.get('job_id')!r}") from exc
        normalized = dict(record)
        normalized['raw_status'] = record['status']
        normalized['canonical_status'] = member.name
        normalized['was_alias'] = record['status'] != member.name
        result.append(normalized)
    return result

incoming = [
    {'system': 'one', 'job_id': 101, 'status': 'busy'},
    {'system': 'two', 'job_id': 102, 'status': 'ran_ok'},
    {'system': 'one', 'job_id': 103, 'status': 'ready'},
]
normalized = normalize_records(incoming)
print(json.dumps(normalized, indent=2))

[
  {
    "system": "one",
    "job_id": 101,
    "status": "busy",
    "raw_status": "busy",
    "canonical_status": "running",
    "was_alias": true
  },
  {
    "system": "two",
    "job_id": 102,
    "status": "ran_ok",
    "raw_status": "ran_ok",
    "canonical_status": "ok",
    "was_alias": true
  },
  {
    "system": "one",
    "job_id": 103,
    "status": "ready",
    "raw_status": "ready",
    "canonical_status": "ready",
    "was_alias": false
  }
]


### Verification 11

Execute the next cell. It uses assertions so a silent pass means the requirements hold.

In [23]:
assert [r['canonical_status'] for r in normalized] == ['running', 'ok', 'ready']
assert [r['was_alias'] for r in normalized] == [True, True, False]
assert incoming[0] == {'system': 'one', 'job_id': 101, 'status': 'busy'}
assert normalized[0]['raw_status'] == 'busy'
try:
    normalize_records([{'job_id': 404, 'status': 'unknown'}])
except ValueError as exc:
    assert '404' in str(exc)
else:
    raise AssertionError('Unknown status must be rejected')
print('PASS 11 — provenance-preserving batch normalization')

PASS 11 — provenance-preserving batch normalization


## Problem 12 — Count canonical states without double counting aliases  
**Level:** Advanced

### Challenge

Given a mixed list of external status strings, build frequency counts by canonical state. Show why iterating all declared names would not be the right way to initialize counters, and return a zero for any canonical state not observed.

### Solution 12: reasoning

Use the member object as a counter key: aliases collapse to one object. Initialize from `ExternalStatus` iteration (four canonical members) and update after parsing each incoming name.

In [24]:
def count_statuses(raw_statuses: Iterable[str]) -> dict[str, int]:
    counts = {member: 0 for member in ExternalStatus}
    for raw in raw_statuses:
        counts[parse_status_name(raw)] += 1
    return {member.name: counts[member] for member in ExternalStatus}

sample_statuses = ['busy', 'processing', 'running', 'ran_ok', 'finished_no_error', 'ready']
status_counts = count_statuses(sample_statuses)
print(status_counts)

{'ready': 1, 'running': 3, 'ok': 2, 'errors': 0}


### Verification 12

Execute the next cell. It uses assertions so a silent pass means the requirements hold.

In [25]:
assert status_counts == {'ready': 1, 'running': 3, 'ok': 2, 'errors': 0}
assert sum(status_counts.values()) == len(sample_statuses)
assert len(status_counts) == len(ExternalStatus) == 4
print('PASS 12 — correct aggregation')

PASS 12 — correct aggregation


## Problem 13 — Handle case and whitespace only when explicitly authorized  
**Level:** Advanced

### Challenge

Implement a configurable parser accepting `(strip, casefold)` flags. Default behavior must remain strict. With both options enabled, strings such as `'  BUSY  '` may normalize. Reject nonstrings even when normalization is enabled.

### Solution 13: reasoning

String cleanup is an input policy, not a property of enum aliasing. Normalize the external token first, then perform declared-name lookup. Keeping strict defaults prevents accidental acceptance of malformed payloads.

In [26]:
def parse_status_configurable(raw: object, *, strip: bool = False,
                              casefold: bool = False) -> ExternalStatus:
    if not isinstance(raw, str):
        raise ValueError(f'Expected status string; got {raw!r}')
    candidate = raw.strip() if strip else raw
    candidate = candidate.casefold() if casefold else candidate
    return parse_status_name(candidate)

print(parse_status_configurable('  BUSY  ', strip=True, casefold=True))

ExternalStatus.running


### Verification 13

Execute the next cell. It uses assertions so a silent pass means the requirements hold.

In [27]:
assert parse_status_configurable('  BUSY  ', strip=True, casefold=True) is ExternalStatus.running
for options in ({}, {'strip': True}, {'casefold': True}):
    try:
        parse_status_configurable('  BUSY  ', **options)
    except ValueError:
        pass
    else:
        raise AssertionError(f'Should reject input with options {options}')
try:
    parse_status_configurable(1, strip=True, casefold=True)
except ValueError:
    pass
else:
    raise AssertionError('Nonstring input accepted')
print('PASS 13 — explicit normalization policy')

PASS 13 — explicit normalization policy


---

# Part IV — Value-based compatibility and defensive validation

Use intentional mechanisms for raw values and catch duplicate-value mistakes at class creation.

## Problem 14 — Use `_missing_` to support external *value* lookup  
**Level:** Expert

### Challenge

Design a string-valued `Enum` whose canonical values are `'ready'`, `'running'`, `'ok'`, and `'errors'`. Support `WireStatus('busy')` and `WireStatus('ran_ok')` using `_missing_`, while rejecting unrelated raw values. Do not invent alias names inside `_missing_`.

### Solution 14: reasoning

`_missing_(cls, value)` is consulted after normal value lookup fails. Use an explicit, immutable-looking mapping of accepted external strings to canonical string values; return the canonical member from the hook. Unlike declared aliases, this handles **value lookup**. Avoid recursive calls on the same unknown string.

In [28]:
class WireStatus(enum.Enum):
    READY = 'ready'
    RUNNING = 'running'
    OK = 'ok'
    ERRORS = 'errors'

    @classmethod
    def _missing_(cls, value: object):
        legacy_to_canonical = {
            'busy': 'running',
            'processing': 'running',
            'finished_no_error': 'ok',
            'ran_ok': 'ok',
            'finished_with_errors': 'errors',
            'errored': 'errors',
        }
        canonical_value = legacy_to_canonical.get(value) if isinstance(value, str) else None
        return cls(canonical_value) if canonical_value is not None else None

print('Value-based conversion:', WireStatus('busy'))
print('Declared names:', tuple(WireStatus.__members__))

Value-based conversion: WireStatus.RUNNING
Declared names: ('READY', 'RUNNING', 'OK', 'ERRORS')


### Verification 14

Execute the next cell. It uses assertions so a silent pass means the requirements hold.

In [29]:
assert WireStatus('busy') is WireStatus.RUNNING
assert WireStatus('ran_ok') is WireStatus.OK
assert WireStatus('ready') is WireStatus.READY
assert 'busy' not in WireStatus.__members__
assert len(WireStatus) == 4
for bad in ('nonsense', None, 12):
    try:
        WireStatus(bad)
    except ValueError:
        pass
    else:
        raise AssertionError(f'Accepted unknown wire value: {bad!r}')
print('PASS 14 — value-based compatibility with _missing_')

PASS 14 — value-based compatibility with _missing_


**Engineering note:** A separate parser is usually clearer when normalization needs context, source-specific rules, audit logs, or rich errors. `_missing_` changes behavior globally for that enum class.

## Problem 15 — Enforce `@unique` and capture the exact failure phase  
**Level:** Advanced

### Challenge

Create a `@enum.unique` class with three different values and another with `WAITING` assigned the same value as `READY`. Verify that the second class definition raises `ValueError` immediately. Make sure the rest of the notebook runs.

### Solution 15: reasoning

`@unique` examines all declared names after the enum class object has been created and rejects aliases. The exception occurs **when executing the decorated class definition**, rather than at Python syntax compilation.

In [30]:
@enum.unique
class StrictStatus(enum.Enum):
    READY = 1
    DONE = 2
    ERROR = 3

try:
    @enum.unique
    class InvalidStrictStatus(enum.Enum):
        READY = 1
        WAITING = 1
        DONE = 2
except ValueError as exc:
    uniqueness_error = str(exc)
    print('Class-creation error:', uniqueness_error)
else:
    raise AssertionError('@unique should reject WAITING')

Class-creation error: duplicate values found in <enum 'InvalidStrictStatus'>: WAITING -> READY


### Verification 15

Execute the next cell. It uses assertions so a silent pass means the requirements hold.

In [31]:
assert len(StrictStatus.__members__) == len(StrictStatus) == 3
assert 'WAITING' in uniqueness_error
assert 'READY' in uniqueness_error
assert 'InvalidStrictStatus' not in globals()
print('PASS 15 — @unique rejects alias definitions at class creation')

PASS 15 — @unique rejects alias definitions at class creation


## Problem 16 — Find unexpected aliases caused by cross-type equality  
**Level:** Expert

### Challenge

Demonstrate that `True == 1` can cause an unintended alias in an ordinary enum. Inspect object identity and canonical names. Then design a better representation that avoids mixing Boolean and integer values in one enum.

### Solution 16: reasoning

An enum compares member values using the values' equality semantics when deciding whether a new declaration duplicates an earlier one. Because Python treats `True` and `1` as equal, a later value `1` may alias a prior `True`. Prefer homogeneous value domains; `@unique` also makes this accident fail at definition time.

In [32]:
class SurprisingValue(enum.Enum):
    ENABLED = True
    RETRY_ONCE = 1
    DISABLED = False
    RETRY_NEVER = 0

@enum.unique
class SafePolicy(enum.Enum):
    ENABLED = 'enabled'
    RETRY_ONCE = 'retry_once'
    DISABLED = 'disabled'
    RETRY_NEVER = 'retry_never'

print('Surprising aliases:', alias_groups(SurprisingValue))

Surprising aliases: {'ENABLED': ('ENABLED', 'RETRY_ONCE'), 'DISABLED': ('DISABLED', 'RETRY_NEVER')}


### Verification 16

Execute the next cell. It uses assertions so a silent pass means the requirements hold.

In [33]:
assert True == 1 and False == 0
assert SurprisingValue.ENABLED is SurprisingValue.RETRY_ONCE
assert SurprisingValue.DISABLED is SurprisingValue.RETRY_NEVER
assert len(SurprisingValue) == 2
assert len(SafePolicy) == 4
assert not has_aliases(SafePolicy)
print('PASS 16 — mixed-value collision and safe redesign')

PASS 16 — mixed-value collision and safe redesign


## Problem 17 — Separate unique codes from accepted legacy spellings  
**Level:** Expert

### Challenge

Build a uniquely valued enum of HTTP-like process stages, with canonical **integer** codes. Keep legacy string spellings in an external mapping rather than aliases. Parse aliases and numeric codes without accidentally accepting `True` as code `1`.

### Solution 17: reasoning

A separate compatibility table can accept legacy strings while `@unique` keeps persisted numeric codes distinct. Explicitly reject Booleans before integer conversion because `bool` subclasses `int` and compares equal to numeric codes.

In [34]:
@enum.unique
class Stage(enum.Enum):
    QUEUED = 10
    RUNNING = 20
    SUCCEEDED = 30
    FAILED = 40

LEGACY_STAGE_NAMES: dict[str, Stage] = {
    'waiting': Stage.QUEUED,
    'busy': Stage.RUNNING,
    'finished_no_error': Stage.SUCCEEDED,
    'errored': Stage.FAILED,
}

def parse_stage(raw: object) -> Stage:
    if isinstance(raw, str):
        try:
            return LEGACY_STAGE_NAMES[raw]
        except KeyError as exc:
            raise ValueError(f'Unknown stage name: {raw!r}') from exc
    if type(raw) is int:
        return Stage(raw)
    raise ValueError(f'Unsupported stage representation: {raw!r}')

print('Legacy string:', parse_stage('busy'))
print('Numeric code:', parse_stage(30))

Legacy string: Stage.RUNNING
Numeric code: Stage.SUCCEEDED


### Verification 17

Execute the next cell. It uses assertions so a silent pass means the requirements hold.

In [35]:
assert parse_stage('busy') is Stage.RUNNING
assert parse_stage(30) is Stage.SUCCEEDED
assert len(Stage.__members__) == len(Stage) == 4
for bad in (True, 1.0, 'RUNNING', 999):
    try:
        parse_stage(bad)
    except ValueError:
        pass
    else:
        raise AssertionError(f'Accepted invalid stage: {bad!r}')
print('PASS 17 — separate compatibility map and strict numeric codes')

PASS 17 — separate compatibility map and strict numeric codes


---

# Part V — Serialization, caching, and API contracts

Explore what aliasing changes when information crosses process boundaries.

## Problem 18 — Round-trip JSON with canonical output and legacy input  
**Level:** Expert

### Challenge

Create `encode_status()` returning a JSON string containing a canonical status name and `decode_status()` accepting JSON records whose status may be any declared external name. Validate the object shape and field type. Demonstrate that encoding an alias writes the canonical spelling.

### Solution 18: reasoning

JSON stores strings rather than enum objects. Encode `.name` intentionally so output is stable and canonical. Parse incoming name tokens through the strict validation function. A round trip preserves member identity, **not** the original external spelling unless that is separately stored.

In [36]:
def encode_status(member: ExternalStatus) -> str:
    if not isinstance(member, ExternalStatus):
        raise TypeError('Expected ExternalStatus')
    return json.dumps({'status': member.name}, sort_keys=True)


def decode_status(payload: str) -> ExternalStatus:
    try:
        obj = json.loads(payload)
    except (TypeError, json.JSONDecodeError) as exc:
        raise ValueError('Invalid status JSON') from exc
    if not isinstance(obj, dict) or 'status' not in obj:
        raise ValueError('Expected JSON object containing a status field')
    return parse_status_name(obj['status'])

encoded_alias = encode_status(ExternalStatus.busy)
print(encoded_alias)

{"status": "running"}


### Verification 18

Execute the next cell. It uses assertions so a silent pass means the requirements hold.

In [37]:
assert encoded_alias == '{"status": "running"}'
assert decode_status(encoded_alias) is ExternalStatus.running
assert decode_status('{"status": "busy"}') is ExternalStatus.running
for bad in ('["ready"]', '{"other": "ready"}', '{"status": 2}', 'not-json'):
    try:
        decode_status(bad)
    except ValueError:
        pass
    else:
        raise AssertionError(f'Accepted invalid JSON: {bad!r}')
print('PASS 18 — canonical JSON round-trip')

PASS 18 — canonical JSON round-trip


## Problem 19 — Preserve external spelling in a frozen value object  
**Level:** Expert

### Challenge

Define an immutable parsed-status object containing both the `ExternalStatus` member and the raw external token. Make it usable in sets. Verify that two raw spellings can refer to the same member while remaining distinguishable as audit records.

### Solution 19: reasoning

An enum member cannot remember which alias spelling was used. A separate frozen dataclass cleanly holds provenance and the canonical member, and can be hashed if its fields are hashable.

In [38]:
@dataclass(frozen=True)
class ParsedStatus:
    canonical: ExternalStatus
    raw: str


def parse_with_provenance(raw: str) -> ParsedStatus:
    return ParsedStatus(canonical=parse_status_name(raw), raw=raw)

record_a = parse_with_provenance('busy')
record_b = parse_with_provenance('processing')
print(record_a, record_b)

ParsedStatus(canonical=<ExternalStatus.running: 'running'>, raw='busy') ParsedStatus(canonical=<ExternalStatus.running: 'running'>, raw='processing')


### Verification 19

Execute the next cell. It uses assertions so a silent pass means the requirements hold.

In [39]:
assert record_a.canonical is record_b.canonical is ExternalStatus.running
assert record_a != record_b
assert len({record_a, record_b}) == 2
assert record_a.raw == 'busy' and record_b.raw == 'processing'
print('PASS 19 — immutable provenance object')

PASS 19 — immutable provenance object


## Problem 20 — Understand dictionary-key collisions caused by aliases  
**Level:** Advanced

### Challenge

Insert values into a dictionary under keys `ExternalStatus.running`, `ExternalStatus.busy`, and `ExternalStatus.processing`. Predict the final number of keys and final value, then implement a counter that intentionally aggregates all three source spellings.

### Solution 20: reasoning

All three keys are the same enum member, so repeated assignment overwrites one dictionary entry. When aggregation is desired, use `Counter` or explicit addition instead of assignment.

In [40]:
overwritten = {}
overwritten[ExternalStatus.running] = 'canonical'
overwritten[ExternalStatus.busy] = 'system-one'
overwritten[ExternalStatus.processing] = 'system-two'

aggregated = Counter(parse_status_name(token)
                     for token in ('running', 'busy', 'processing', 'ran_ok'))
print('Overwritten:', overwritten)
print('Aggregated:', {member.name: count for member, count in aggregated.items()})

Overwritten: {<ExternalStatus.running: 'running'>: 'system-two'}
Aggregated: {'running': 3, 'ok': 1}


### Verification 20

Execute the next cell. It uses assertions so a silent pass means the requirements hold.

In [41]:
assert len(overwritten) == 1
assert overwritten[ExternalStatus.running] == 'system-two'
assert aggregated[ExternalStatus.running] == 3
assert aggregated[ExternalStatus.ok] == 1
assert len(aggregated) == 2
print('PASS 20 — dictionary collision vs intentional aggregation')

PASS 20 — dictionary collision vs intentional aggregation


---

# Part VI — Enum variants and non-obvious edge cases

Know when alias intuition holds and when a different enum type introduces additional concerns.

## Problem 21 — Contrast `Enum` and `IntEnum` equality  
**Level:** Expert

### Challenge

Define two unrelated `IntEnum` classes with identical integer values plus one ordinary `Enum`. Compare a member to the integer and across the two `IntEnum` classes. Demonstrate a safer type-check before accepting an enum from a public API.

### Solution 21: reasoning

Ordinary `Enum` members do not equal their raw integer values. `IntEnum`, by design, also behaves like an integer, so cross-class equality can be true for equal numeric values even when identity is false. Validate the enum class explicitly at boundaries.

In [42]:
class FirstCode(enum.IntEnum):
    OK = 200
    SUCCESS = 200

class SecondCode(enum.IntEnum):
    OK = 200

class PlainCode(enum.Enum):
    OK = 200


def require_first_code(value: object) -> FirstCode:
    if not isinstance(value, FirstCode):
        raise TypeError('Expected FirstCode specifically')
    return value

print('IntEnum == integer:', FirstCode.OK == 200)
print('Different IntEnum classes compare:', FirstCode.OK == SecondCode.OK)

IntEnum == integer: True
Different IntEnum classes compare: True


### Verification 21

Execute the next cell. It uses assertions so a silent pass means the requirements hold.

In [43]:
assert FirstCode.OK is FirstCode.SUCCESS
assert FirstCode.OK == 200
assert FirstCode.OK == SecondCode.OK
assert FirstCode.OK is not SecondCode.OK
assert PlainCode.OK != 200
assert require_first_code(FirstCode.SUCCESS) is FirstCode.OK
for wrong in (200, SecondCode.OK, PlainCode.OK):
    try:
        require_first_code(wrong)
    except TypeError:
        pass
    else:
        raise AssertionError(f'Accepted wrong type {wrong!r}')
print('PASS 21 — IntEnum interoperability hazards')

PASS 21 — IntEnum interoperability hazards


## Problem 22 — Explain aliases versus composite `Flag` values  
**Level:** Expert

### Challenge

Create a permission `Flag` with `READ`, `WRITE`, a same-valued name `VIEW`, and a composite `READ_WRITE = READ | WRITE`. Compare iteration and `__members__`. Explain why the composite is not a distinct single-bit flag and why absence from `list()` does not automatically mean a name is an alias.

### Solution 22: reasoning

A same-valued name such as `VIEW` is a true alias of `READ`. A combination such as `READ_WRITE` is a valid named composite, with a distinct value and object. `Flag` iteration yields canonical **single-bit** members, normally omitting composites as well as aliases. To detect aliases use `.name` and `__members__`, not the difference between `list(Flag)` and `__members__` alone.

In [44]:
class Permission(enum.Flag):
    READ = enum.auto()
    VIEW = READ
    WRITE = enum.auto()
    READ_WRITE = READ | WRITE

print('Flag iteration:', list(Permission))
print('All declared names:', list(Permission.__members__))
print('Alias inventory:', alias_groups(Permission))

Flag iteration: [<Permission.READ: 1>, <Permission.WRITE: 2>]
All declared names: ['READ', 'VIEW', 'WRITE', 'READ_WRITE']
Alias inventory: {'READ': ('READ', 'VIEW'), 'WRITE': ('WRITE',), 'READ_WRITE': ('READ_WRITE',)}


### Verification 22

Execute the next cell. It uses assertions so a silent pass means the requirements hold.

In [45]:
assert Permission.VIEW is Permission.READ
assert Permission.READ_WRITE == (Permission.READ | Permission.WRITE)
assert Permission.READ_WRITE is not Permission.READ
assert set(Permission.__members__) == {'READ', 'VIEW', 'WRITE', 'READ_WRITE'}
assert [m.name for m in Permission] == ['READ', 'WRITE']
assert alias_groups(Permission)['READ'] == ('READ', 'VIEW')
assert Permission.__members__['READ_WRITE'].name == 'READ_WRITE'
print('PASS 22 — Flag composite is not a duplicate-value alias')

PASS 22 — Flag composite is not a duplicate-value alias


**Engineering note:** For ordinary `Enum`, iteration omits only aliases; `Flag` has additional iteration rules for composite and zero-valued flags.

---

# Part VII — Two capstone problems

Combine parsing, auditing, precise errors, immutability, and tests in realistic workflows.

## Problem 23 — Build a source-aware compatibility registry  
**Level:** Expert

### Challenge

Implement a registry with explicitly supported tokens per system. System A accepts `ready`, `busy`, `finished_no_error`, `finished_with_errors`; system B accepts `ready`, `processing`, `ran_ok`, `errored`. Also support our own four canonical names under system `internal`. Reject a valid token when used with the **wrong** system. Return provenance (`system`, raw token, canonical enum). Validate the registry at construction: no unknown token, no duplicate token within a system, and no empty system.

### Solution 23: reasoning

Global aliases alone do not constrain a token by upstream system. A source-aware registry should map each system's allowed spellings to enum members, validate configuration once, and return an immutable record. Restricting each source prevents accidental cross-system acceptance and improves diagnostics.

In [46]:
SYSTEM_TOKENS = {
    'system_a': ('ready', 'busy', 'finished_no_error', 'finished_with_errors'),
    'system_b': ('ready', 'processing', 'ran_ok', 'errored'),
    'internal': ('ready', 'running', 'ok', 'errors'),
}

@dataclass(frozen=True)
class StatusEvent:
    system: str
    raw: str
    status: ExternalStatus

class StatusRegistry:
    def __init__(self, allowed: Mapping[str, Iterable[str]]) -> None:
        self._lookups: dict[str, dict[str, ExternalStatus]] = {}
        for system, supplied_names in allowed.items():
            if not isinstance(system, str) or not system:
                raise ValueError('A system name must be a nonempty string')
            names = tuple(supplied_names)
            if not names:
                raise ValueError(f'No names configured for {system!r}')
            if len(names) != len(set(names)):
                raise ValueError(f'Duplicate token configured for {system!r}')
            try:
                lookup = {name: parse_status_name(name) for name in names}
            except ValueError as exc:
                raise ValueError(f'Invalid configuration for {system!r}') from exc
            self._lookups[system] = lookup

    def parse(self, system: str, raw: str) -> StatusEvent:
        if system not in self._lookups:
            raise ValueError(f'Unknown system: {system!r}')
        try:
            canonical = self._lookups[system][raw]
        except (KeyError, TypeError) as exc:
            raise ValueError(f'Invalid token {raw!r} for system {system!r}') from exc
        return StatusEvent(system=system, raw=raw, status=canonical)

registry = StatusRegistry(SYSTEM_TOKENS)
print(registry.parse('system_a', 'busy'))
print(registry.parse('system_b', 'ran_ok'))

StatusEvent(system='system_a', raw='busy', status=<ExternalStatus.running: 'running'>)
StatusEvent(system='system_b', raw='ran_ok', status=<ExternalStatus.ok: 'ok'>)


### Verification 23

Execute the next cell. It uses assertions so a silent pass means the requirements hold.

In [47]:
assert registry.parse('system_a', 'busy').status is ExternalStatus.running
assert registry.parse('system_b', 'ran_ok').status is ExternalStatus.ok
assert registry.parse('internal', 'running').status is ExternalStatus.running
assert registry.parse('system_a', 'busy').raw == 'busy'
for source, token in [('system_a', 'processing'), ('system_b', 'busy'),
                      ('internal', 'ran_ok'), ('unknown', 'ready')]:
    try:
        registry.parse(source, token)
    except ValueError:
        pass
    else:
        raise AssertionError(f'Accepted invalid pair {(source, token)!r}')
for bad_config in ({'x': ()}, {'x': ('ready', 'ready')},
                   {'x': ('not_real',)}, {'': ('ready',)}):
    try:
        StatusRegistry(bad_config)
    except ValueError:
        pass
    else:
        raise AssertionError(f'Accepted invalid registry config: {bad_config!r}')
print('PASS 23 — source-aware, validated compatibility registry')

PASS 23 — source-aware, validated compatibility registry


**Engineering note:** This design is appropriate when external spellings are source-specific rather than universally interchangeable.

## Problem 24 — Create a migration audit with exhaustive regression tests  
**Level:** Expert

### Challenge

Given source-labelled `(system, token)` pairs, produce a report with four canonical-state counts, counts of each actually observed legacy spelling, and a normalized event list. Preserve event order. Test every configured token exhaustively; ensure no input is lost or double counted, and prove source restrictions still apply.

### Solution 24: reasoning

Reuse the registry instead of duplicating normalization logic. Count enum member objects for canonical totals and `(system, token)` pairs for raw-spelling audit totals. For deterministic reports, iterate canonical members for the totals and sort legacy keys for serialized dictionary output. Exhaustive tests cover every supported registry token without third-party test packages.

In [48]:
def migration_audit(pairs: Iterable[tuple[str, str]],
                    registry: StatusRegistry) -> dict[str, object]:
    events = [registry.parse(system, token) for system, token in pairs]
    canonical_counter = Counter(event.status for event in events)
    legacy_counter = Counter((event.system, event.raw) for event in events)
    return {
        'total': len(events),
        'canonical_counts': {member.name: canonical_counter[member]
                             for member in ExternalStatus},
        'spelling_counts': {f'{system}:{token}': count
                            for (system, token), count in sorted(legacy_counter.items())},
        'events': [{'system': event.system, 'raw': event.raw,
                    'canonical': event.status.name} for event in events],
    }

all_configured_pairs = [(system, token) for system, tokens in SYSTEM_TOKENS.items()
                        for token in tokens]
complete_report = migration_audit(all_configured_pairs, registry)
print(json.dumps(complete_report, indent=2))

{
  "total": 12,
  "canonical_counts": {
    "ready": 3,
    "running": 3,
    "ok": 3,
    "errors": 3
  },
  "spelling_counts": {
    "internal:errors": 1,
    "internal:ok": 1,
    "internal:ready": 1,
    "internal:running": 1,
    "system_a:busy": 1,
    "system_a:finished_no_error": 1,
    "system_a:finished_with_errors": 1,
    "system_a:ready": 1,
    "system_b:errored": 1,
    "system_b:processing": 1,
    "system_b:ran_ok": 1,
    "system_b:ready": 1
  },
  "events": [
    {
      "system": "system_a",
      "raw": "ready",
      "canonical": "ready"
    },
    {
      "system": "system_a",
      "raw": "busy",
      "canonical": "running"
    },
    {
      "system": "system_a",
      "raw": "finished_no_error",
      "canonical": "ok"
    },
    {
      "system": "system_a",
      "raw": "finished_with_errors",
      "canonical": "errors"
    },
    {
      "system": "system_b",
      "raw": "ready",
      "canonical": "ready"
    },
    {
      "system": "system_b",
      

### Verification 24

Execute the next cell. It uses assertions so a silent pass means the requirements hold.

In [49]:
assert complete_report['total'] == 12
assert complete_report['canonical_counts'] == {'ready': 3, 'running': 3,
                                                'ok': 3, 'errors': 3}
assert sum(complete_report['canonical_counts'].values()) == complete_report['total']
assert sum(complete_report['spelling_counts'].values()) == complete_report['total']
assert [(item['system'], item['raw']) for item in complete_report['events']] == all_configured_pairs
for system, token in all_configured_pairs:
    one = migration_audit([(system, token)], registry)
    assert one['total'] == 1
    assert sum(one['canonical_counts'].values()) == 1
    assert one['events'][0]['canonical'] == registry.parse(system, token).status.name
for system, bad_token in [('system_a', 'ran_ok'), ('system_b', 'finished_no_error')]:
    try:
        migration_audit([(system, bad_token)], registry)
    except ValueError:
        pass
    else:
        raise AssertionError(f'Accepted cross-source token {bad_token!r}')
print('PASS 24 — exhaustive source tokens and reconciliation invariants')

PASS 24 — exhaustive source tokens and reconciliation invariants


---

# Reference and self-check

Consolidated practices and a final notebook-wide regression cell.

## Best-practice checklist

1. Decide whether synonyms are merely **alternate names for one state**. If yes, aliases are suitable; if meanings differ, do not alias.
2. Put the intended canonical name **first**. Do not accidentally change persisted `.name` or `repr` by reordering declarations.
3. Distinguish **name lookup** (`E['alias']`) from **value lookup** (`E(value)`); use a parser or `_missing_` for unrecognized external **values**.
4. Iterate with `for member in E` for distinct canonical members; inspect `E.__members__` for every declared name.
5. Use `@unique` when values must be one-to-one with names, and apply it as part of the definition.
6. Prefer explicit normalization policies and source-aware mappings at API boundaries. Reject unknown inputs and avoid over-permissive defaults.
7. Preserve raw tokens independently when analytics, telemetry, or debugging require knowing the original spelling.
8. Keep storage, display, and transport policies explicit: choose `.name` or `.value` intentionally, and test schema compatibility.
9. Do not mix Booleans and integers as enum values; use explicit type checks when numeric codes are accepted.
10. Understand `IntEnum` numeric interoperability and `Flag` composite iteration before reusing generic enum algorithms.

### Quick reference

| Expression | Meaning | Includes aliases? |
|---|---|---|
| `E['ALIAS']` | Lookup by declared name | Yes |
| `E(stored_value)` | Lookup by stored value | Returns canonical object; no raw name lookup |
| `list(E)` | Unique members (`Enum`) | No |
| `len(E)` | Number of unique members (`Enum`) | No |
| `E.__members__` | All declared names → member objects | Yes |
| `member.name` | Canonical declared name | Canonical only |
| `@enum.unique` | Reject duplicate-valued declarations | Rejects them at class creation |

**Version note:** Tested on current Python 3 with 3.10+ syntax. The lesson's older Python 3.6 examples use the same basic alias semantics, but this notebook intentionally uses modern type hints.

In [50]:
# Regression summary: running this cell after all prior cells verifies key invariants.
assert len(PolygonSides.__members__) == 4 and len(PolygonSides) == 2
assert alias_groups(PolygonSides)['RECTANGLE'] == ('RECTANGLE', 'SQUARE', 'RHOMBUS')
assert len(ExternalStatus.__members__) == 10 and len(ExternalStatus) == 4
assert parse_status_name('processing') is ExternalStatus.running
assert WireStatus('processing') is WireStatus.RUNNING
assert len(SafePolicy) == 4 and not has_aliases(SafePolicy)
assert len(complete_report['events']) == 12
assert sum(complete_report['canonical_counts'].values()) == 12
print('ALL 24 PROBLEM SOLUTIONS COMPLETED; NOTEBOOK-WIDE REGRESSION CHECKS PASSED.')

ALL 24 PROBLEM SOLUTIONS COMPLETED; NOTEBOOK-WIDE REGRESSION CHECKS PASSED.


## Optional extension prompts (try independently)

- Extend the registry with a third upstream system that has five legacy spellings but only four distinct states. Decide whether two distinct upstream spellings can map to one canonical state and demonstrate it in tests.
- Add a `format_error` method listing only the valid names for a given system; avoid listing unrelated systems' tokens.
- Define a versioned JSON envelope with `schema_version` and `raw_status`; test migration from a legacy envelope to a canonical envelope without destroying provenance.
- Create a deliberately invalid enum with a duplicate value that crosses `True`/`1`, and confirm that `@unique` catches the problem at class creation.
- Add randomized tests using a fixed seed that choose from configured names, then verify total counts and event order for 500 records.

**End of the single-file notebook.** All exercises and their complete solutions are contained here; no supplementary files are needed.